## Healthcare Nursing Documentation Burden — Fabric IQ

### Create IQ Ontology from RDF/OWL Definition

This notebook creates the **NursingDocBurdenOntology** in Fabric IQ from an inline RDF/OWL definition
that models the nursing documentation burden domain.

> **Note – Concept Mapping**
>
> - Classes → Entity Type
> - Data Properties → Properties
> - Object Properties → Relationship Type

> **Note – Known Limitations**
>
> - Multiple domains/ranges on object properties are automatically split into separate relationship types
> - `rdfs:subClassOf` relationships are not processed — all classes are independent types
> - Maximum 1000 classes per ontology
> - RDF-based ontologies create entity types and relationships but do **not** include data bindings or contextualizations — use the Package approach for full bindings

**Ontology Entities Defined:**
- Nurse, Patient, HospitalUnit, DocumentationType, Medication, Diagnosis

**Real-Time Streaming Event Classes (CTX-007–011):**
- RTLSLocationPing, EHRClickstream, NurseCallEvent, BCMAScanEvent, ClinicalAlert

**Relationships Defined:**
- assignedTo, admittedTo, hasDiagnosis, caresFor, prescribedFor, tracksNurse, alertsFor, scansFor

In [ ]:
# Install Fabric IQ Ontology Accelerator Package
%pip install /lakehouse/default/Files/fabriciq_ontology_accelerator-0.1.0-py3-none-any.whl --q

In [ ]:
import sempy.fabric as fabric
import json, requests, re
from urllib.parse import urlparse
from fabricontology import create_ontology_item, generate_definition_from_rdf

# ── Zero Trust: URL allowlist for RDF sources ──────────────────────────
_ALLOWED_RDF_DOMAINS = {
    "raw.githubusercontent.com", "github.com",
    "w3.org", "www.w3.org",
    "schema.org", "purl.org", "xmlns.com",
    "ontology.iq.microsoft.com", "fabric.microsoft.com",
}

def _is_allowed_url(url):
    """Validate URL against allowlist; block private IPs and non-HTTPS."""
    try:
        parsed = urlparse(url)
        if parsed.scheme not in ("https",):
            return False, "Only HTTPS URLs are allowed"
        host = parsed.hostname or ""
        # Block private / loopback ranges
        if re.match(r"^(10\.|172\.(1[6-9]|2[0-9]|3[01])\.|192\.168\.|127\.|0\.)", host):
            return False, "Private/loopback IPs are blocked"
        if any(host == d or host.endswith("." + d) for d in _ALLOWED_RDF_DOMAINS):
            return True, "OK"
        return False, f"Domain '{host}' not in RDF allowlist"
    except Exception as e:
        return False, str(e)

def get_rdf_content_from_url(rdf_url):
    """Fetch RDF content from a validated URL (Zero Trust: allowlist + timeout)."""
    allowed, reason = _is_allowed_url(rdf_url)
    if not allowed:
        raise ValueError(f"ZT URL validation failed for '{rdf_url}': {reason}")
    print(f"  ✓ URL validated: {rdf_url}")
    response = requests.get(rdf_url, timeout=30)
    response.raise_for_status()
    response.encoding = "utf-8"
    return response.text

def get_rdf_content_from_file(file_path):
    """Read RDF content from a local file."""
    with open(file_path, "r", encoding="utf-8") as file:
        return file.read()

In [ ]:
# ── Inline RDF/OWL definition for Nursing Documentation Burden ─────────
#
# This defines the healthcare ontology using OWL classes, data properties,
# and object properties that map to Fabric IQ entity types, properties,
# and relationship types respectively.

rdf_content = """
<?xml version="1.0" encoding="UTF-8"?>
<rdf:RDF
    xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#"
    xmlns:rdfs="http://www.w3.org/2000/01/rdf-schema#"
    xmlns:owl="http://www.w3.org/2002/07/owl#"
    xmlns:xsd="http://www.w3.org/2001/XMLSchema#"
    xmlns:ndb="http://fabric.microsoft.com/ontologies/nursing-doc-burden#">

  <!-- ================================================================ -->
  <!-- Ontology Declaration                                              -->
  <!-- ================================================================ -->
  <owl:Ontology rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden">
    <rdfs:label>Nursing Documentation Burden Ontology</rdfs:label>
    <rdfs:comment>Models the healthcare nursing documentation burden domain for Fabric IQ. Covers nurses, patients, hospital units, documentation types, medications, and diagnoses.</rdfs:comment>
  </owl:Ontology>

  <!-- ================================================================ -->
  <!-- Classes (→ Entity Types)                                          -->
  <!-- ================================================================ -->
  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse">
    <rdfs:label>Nurse</rdfs:label>
    <rdfs:comment>A registered nurse responsible for patient care and clinical documentation.</rdfs:comment>
  </owl:Class>

  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient">
    <rdfs:label>Patient</rdfs:label>
    <rdfs:comment>A patient admitted to a hospital unit receiving nursing care.</rdfs:comment>
  </owl:Class>

  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#HospitalUnit">
    <rdfs:label>HospitalUnit</rdfs:label>
    <rdfs:comment>A hospital unit such as Med-Surg, ICU, or Emergency.</rdfs:comment>
  </owl:Class>

  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#DocumentationType">
    <rdfs:label>DocumentationType</rdfs:label>
    <rdfs:comment>A type of clinical documentation such as assessments, medication records, or progress notes.</rdfs:comment>
  </owl:Class>

  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Medication">
    <rdfs:label>Medication</rdfs:label>
    <rdfs:comment>A medication that can be administered to patients.</rdfs:comment>
  </owl:Class>

  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Diagnosis">
    <rdfs:label>Diagnosis</rdfs:label>
    <rdfs:comment>A clinical diagnosis with ICD-10 classification.</rdfs:comment>
  </owl:Class>

  <!-- ================================================================ -->
  <!-- Data Properties (→ Properties on Entity Types)                    -->
  <!-- ================================================================ -->

  <!-- Nurse properties -->
  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#nurseId">
    <rdfs:label>nurseId</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#firstName">
    <rdfs:label>firstName</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#lastName">
    <rdfs:label>lastName</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#role">
    <rdfs:label>role</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#credentials">
    <rdfs:label>credentials</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#yearsExperience">
    <rdfs:label>yearsExperience</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#certifications">
    <rdfs:label>certifications</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#shiftPreference">
    <rdfs:label>shiftPreference</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <!-- Patient properties -->
  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#patientId">
    <rdfs:label>patientId</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#patientFirstName">
    <rdfs:label>patientFirstName</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#patientLastName">
    <rdfs:label>patientLastName</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#age">
    <rdfs:label>age</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#gender">
    <rdfs:label>gender</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#acuityLevel">
    <rdfs:label>acuityLevel</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#insuranceType">
    <rdfs:label>insuranceType</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <!-- HospitalUnit properties -->
  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#unitId">
    <rdfs:label>unitId</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#HospitalUnit"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#unitName">
    <rdfs:label>unitName</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#HospitalUnit"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#unitType">
    <rdfs:label>unitType</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#HospitalUnit"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#bedCount">
    <rdfs:label>bedCount</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#HospitalUnit"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#targetNurseRatio">
    <rdfs:label>targetNurseRatio</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#HospitalUnit"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#ehrSystem">
    <rdfs:label>ehrSystem</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#HospitalUnit"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <!-- DocumentationType properties -->
  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#docTypeId">
    <rdfs:label>docTypeId</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#DocumentationType"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#docTypeName">
    <rdfs:label>docTypeName</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#DocumentationType"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#category">
    <rdfs:label>category</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#DocumentationType"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#avgTimeMinutes">
    <rdfs:label>avgTimeMinutes</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#DocumentationType"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#isDuplicative">
    <rdfs:label>isDuplicative</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#DocumentationType"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <!-- Medication properties -->
  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#medicationId">
    <rdfs:label>medicationId</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Medication"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#medicationName">
    <rdfs:label>medicationName</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Medication"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#drugClass">
    <rdfs:label>drugClass</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Medication"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#route">
    <rdfs:label>route</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Medication"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#highAlert">
    <rdfs:label>highAlert</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Medication"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <!-- Diagnosis properties -->
  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#diagnosisId">
    <rdfs:label>diagnosisId</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Diagnosis"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#diagnosisName">
    <rdfs:label>diagnosisName</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Diagnosis"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#icd10Code">
    <rdfs:label>icd10Code</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Diagnosis"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#typicalLosDays">
    <rdfs:label>typicalLosDays</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Diagnosis"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#diagnosisAcuityLevel">
    <rdfs:label>diagnosisAcuityLevel</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Diagnosis"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#avgDailyDocEvents">
    <rdfs:label>avgDailyDocEvents</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Diagnosis"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <!-- ================================================================ -->
  <!-- Object Properties (→ Relationship Types)                          -->
  <!-- ================================================================ -->

  <owl:ObjectProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#assignedTo">
    <rdfs:label>assignedTo</rdfs:label>
    <rdfs:comment>Nurse is assigned to a HospitalUnit</rdfs:comment>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
    <rdfs:range rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#HospitalUnit"/>
  </owl:ObjectProperty>

  <owl:ObjectProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#admittedTo">
    <rdfs:label>admittedTo</rdfs:label>
    <rdfs:comment>Patient is admitted to a HospitalUnit</rdfs:comment>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
    <rdfs:range rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#HospitalUnit"/>
  </owl:ObjectProperty>

  <owl:ObjectProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#hasDiagnosis">
    <rdfs:label>hasDiagnosis</rdfs:label>
    <rdfs:comment>Patient has a clinical Diagnosis</rdfs:comment>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
    <rdfs:range rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Diagnosis"/>
  </owl:ObjectProperty>

  <owl:ObjectProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#caresFor">
    <rdfs:label>caresFor</rdfs:label>
    <rdfs:comment>Nurse provides care for a Patient during a shift</rdfs:comment>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
    <rdfs:range rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
  </owl:ObjectProperty>

  <owl:ObjectProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#prescribedFor">
    <rdfs:label>prescribedFor</rdfs:label>
    <rdfs:comment>Medication is commonly prescribed for a Diagnosis</rdfs:comment>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Medication"/>
    <rdfs:range rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Diagnosis"/>
  </owl:ObjectProperty>

  <!-- ================================================================ -->
  <!-- Real-Time Streaming Event Classes (CTX-007 through CTX-011)       -->
  <!-- ================================================================ -->

  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#RTLSLocationPing">
    <rdfs:label>RTLSLocationPing</rdfs:label>
    <rdfs:comment>Real-time location system ping from a nurse badge, tracking zone-level movement within a hospital unit.</rdfs:comment>
  </owl:Class>

  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#EHRClickstream">
    <rdfs:label>EHRClickstream</rdfs:label>
    <rdfs:comment>Sub-second UX interaction event in the EHR system — clicks, keystrokes, idle, errors, and navigation.</rdfs:comment>
  </owl:Class>

  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#NurseCallEvent">
    <rdfs:label>NurseCallEvent</rdfs:label>
    <rdfs:comment>A nurse call, bed alarm, or clinical alert that may interrupt documentation workflow.</rdfs:comment>
  </owl:Class>

  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#BCMAScanEvent">
    <rdfs:label>BCMAScanEvent</rdfs:label>
    <rdfs:comment>Barcode-assisted medication administration scan event — patient wristband or medication barcode scan with success/failure/override tracking.</rdfs:comment>
  </owl:Class>

  <owl:Class rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#ClinicalAlert">
    <rdfs:label>ClinicalAlert</rdfs:label>
    <rdfs:comment>Clinical decision support alert that fires in the EHR — tracks alert fatigue, actionability, and documentation disruption.</rdfs:comment>
  </owl:Class>

  <!-- Key data properties for streaming event classes -->

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#zoneType">
    <rdfs:label>zoneType</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#RTLSLocationPing"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#dwellTimeSeconds">
    <rdfs:label>dwellTimeSeconds</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#RTLSLocationPing"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#eventType">
    <rdfs:label>eventType</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#EHRClickstream"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#ehrModule">
    <rdfs:label>ehrModule</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#EHRClickstream"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#interruptedDocumentation">
    <rdfs:label>interruptedDocumentation</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#NurseCallEvent"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#documentationResumeDelaySeconds">
    <rdfs:label>documentationResumeDelaySeconds</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#NurseCallEvent"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#scanResult">
    <rdfs:label>scanResult</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#BCMAScanEvent"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#overrideReason">
    <rdfs:label>overrideReason</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#BCMAScanEvent"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#fatigueScore">
    <rdfs:label>fatigueScore</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#ClinicalAlert"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#documentationDelaySeconds">
    <rdfs:label>documentationDelaySeconds</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#ClinicalAlert"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#integer"/>
  </owl:DatatypeProperty>

  <owl:DatatypeProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#wasActionable">
    <rdfs:label>wasActionable</rdfs:label>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#ClinicalAlert"/>
    <rdfs:range rdf:resource="http://www.w3.org/2001/XMLSchema#string"/>
  </owl:DatatypeProperty>

  <!-- Object properties for streaming events -->

  <owl:ObjectProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#tracksNurse">
    <rdfs:label>tracksNurse</rdfs:label>
    <rdfs:comment>Links a streaming event to the Nurse it tracks</rdfs:comment>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#RTLSLocationPing"/>
    <rdfs:range rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Nurse"/>
  </owl:ObjectProperty>

  <owl:ObjectProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#alertsFor">
    <rdfs:label>alertsFor</rdfs:label>
    <rdfs:comment>Links a clinical alert to the Patient it concerns</rdfs:comment>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#ClinicalAlert"/>
    <rdfs:range rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Patient"/>
  </owl:ObjectProperty>

  <owl:ObjectProperty rdf:about="http://fabric.microsoft.com/ontologies/nursing-doc-burden#scansFor">
    <rdfs:label>scansFor</rdfs:label>
    <rdfs:comment>Links a BCMA scan to the Medication being administered</rdfs:comment>
    <rdfs:domain rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#BCMAScanEvent"/>
    <rdfs:range rdf:resource="http://fabric.microsoft.com/ontologies/nursing-doc-burden#Medication"/>
  </owl:ObjectProperty>

</rdf:RDF>
"""

rdf_format = "xml"

In [ ]:
# ── Generate ontology definition from RDF and create in Fabric ─────────
workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')

ontology_item_name = "NursingDocBurdenOntology_RDF"

ontology_definition, entity_types, relationship_types = generate_definition_from_rdf(
    rdf_content=rdf_content,
    rdf_format=rdf_format,
    ontology_name=ontology_item_name,
)

print(f"Entity Types ({len(entity_types)}): {list(entity_types.keys())}")
print(f"Relationship Types ({len(relationship_types)}): {list(relationship_types.keys())}")

try:
    response = create_ontology_item(
        workspace_id=workspace_id,
        access_token=access_token,
        ontology_item_name=ontology_item_name,
        ontology_definition=ontology_definition,
    )
    print(f"\n[AUDIT] ONTOLOGY_CREATE | target={ontology_item_name} | status={response.status_code}")
    print(json.dumps(response.json(), indent=2))
except Exception as e:
    print(f"\n[AUDIT] ONTOLOGY_CREATE | target={ontology_item_name} | status=ERROR | detail={e}")
    raise

### Notes

- The RDF approach creates **entity types and relationships only** — no data bindings or contextualizations.
- To get full bindings (Lakehouse Delta + Eventhouse KQL), use `Healthcare_Create_Ontology_from_Package.ipynb` with the `.iq` package instead.
- You can combine both approaches: create the ontology structure via RDF, then manually add data bindings and contextualizations.
- The 5 streaming event classes (RTLSLocationPing, EHRClickstream, NurseCallEvent, BCMAScanEvent, ClinicalAlert) model real-time data sources that feed Eventhouse via Real-Time Intelligence Eventstreams.

### Cross-Industry Reusability

This RDF ontology can be adapted for other domains by renaming classes:

| Healthcare | Manufacturing | Retail | Finance |
|-----------|--------------|--------|--------|
| Nurse → | Operator | Associate | Analyst |
| Patient → | WorkOrder | Customer | Account |
| HospitalUnit → | ProductionLine | StoreSection | Department |
| DocumentationType → | InspectionType | TransactionType | ReportType |
| Medication → | Component | Product | Instrument |
| Diagnosis → | DefectCategory | ReturnReason | RiskCategory |
| RTLSLocationPing → | AssetTracking | StoreTraffic | FloorPresence |
| EHRClickstream → | MESClickstream | POSClickstream | TradingClickstream |
| NurseCallEvent → | MachineAlarm | CustomerRequest | ClientEscalation |
| BCMAScanEvent → | QRScan | BarcodeScan | DocumentScan |
| ClinicalAlert → | SafetyAlert | FraudAlert | ComplianceAlert |